# Interpretability for LLMs


## A mathematical framework
[original thread](https://transformer-circuits.pub/2021/framework/index.html)

### Residuals

One of the main features of the high level architecture of a transformer is that each layer adds its results into what we call the “residual stream.” The residual stream is simply the sum of the output of all the previous layers and the original embedding. We generally think of the residual stream as a communication channel, since it doesn't do any processing itself and all layers communicate through it. 

<img src="../data/llm.png" height="480">

### Virtual Weights

An especially useful consequence of the residual stream being linear is that one can think of implicit "virtual weights" directly connecting any pair of layers (even those separated by many other layers), by multiplying out their interactions through the residual stream. These virtual weights are the product of the output weights of one layer with the input weights 5 of another (ie. $W_I^2W_O^1W_I^2​W_O^1$​ ), and describe the extent to which a later layer reads in the information written by a previous layer.

<img src="../data/residuals.png" height="480">

## Auto-encoders

_Autoencoders_ have a special role in interpretability. These self-encoding algorithms simply learn to copy input nodes to output nodes. In other words, they learn the identity function. Mathematically, this is very interesting: a typical example of an identity is the identity matrix, and as we know from linear algebra, all invertible matrices are reducible to precisely $I$. From that alone, we can expect that the system can learn all bijective linear transformations! We can also have fewer output than input nodes, thereby forcing the network to learn latent representations.

Superposition (in neural networks)
> Superposition is when a model represents more than n features in an n-dimensional activation space. That is, features still correspond to directions, but the set of interpretable directions is larger than the number of dimensions.

(The definition according to Neel Nanda, Google DeepMind)

This does not align with the mathematical definition of superposition, but instead corresponds to what in mathematics is called an embedding. Obvious naming conflicts! Complex numbers are an example of what this definition of superposition encompasses. In a matrix of complex numbers, there is an additional “latent” dimension: the complex plane. Quaternions have three latent dimensions. NN superpositions have some $l$ latent dimensions.

$(c_1,c_2, ...)$ =  ($a_1+b_1i$ , $a_2+b_2i$, ... ) -> $\begin{bmatrix}a_1 & b_1 \\ a_2 & b_2 \\ : & :  \\\end{bmatrix}\begin{bmatrix}1 \\ i\end{bmatrix}$


A Sparse Autoencoder (SAE) tends to activate separate nodes in the output layer for clusters of activations in the input — which is useful for feature separation in interpretability.

<img src="../data/6303_blog-visuals-auto-encoders_1500X800.png" height="480">

By modifying the optimization objective to favor sparse representations, we obtain a Sparse Autoencoder. One approach is to apply $\mathcal{l}_1$ regularization between the layers of the network — this would lead to sparse representations as less relevant parameters are driven toward zero. In practice, much better results are achieved using Kullback–Leibler divergence:
\begin{equation*}
D(p || q) = p \log \frac{p}{q} + (1-p) \log \frac{1-p}{1-q}
\end{equation*}



So how do we know these latent feature vectors mapped out by SAEs are real? _Steering_ is a relatively new technique that can test feature vectors. 

- https://neuronpedia.org/



In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers
transformers.logging.set_verbosity_error()
import torch
import os 
#os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
device = "cuda"
dtype = torch.bfloat16

llm = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", dtype=dtype, device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
template = [{"role":"user", "content":"What are some sights to see in your lifetime?"}]
input_ids = tokenizer.apply_chat_template(template, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
outputs = llm.generate(**input_ids, max_new_tokens=256, do_sample=False, repetition_penalty=1.1)

In [ ]:
print(tokenizer.decode(outputs[0]))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

What are some sights to see in your lifetime?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

There are countless incredible sights to see around the world, and what's considered a "must-see" can vary depending on personal interests and preferences. Here are some of the most iconic and awe-inspiring destinations:

**Natural Wonders**

1. **Grand Canyon (USA)**: One of the most spectacular natural wonders in the United States.
2. **Aurora Borealis (Northern Lights) (Arctic/Norway)**: Witness the breathtaking display of colored lights dancing across the night sky.
3. **Great Barrier Reef (Australia)**: Explore the world's largest coral reef system, home to an incredible array of marine life.
4. **Victoria Falls (Zambia/Zimbabwe)**: Marvel at the sheer power and beauty of one of the world's largest waterfall

In [ ]:
import json
import torch
with open("../Data/llama3.1-8b-it_15-resid-post-aa_23443.json", 'r') as fd:
    feature = json.load(fd)
v = torch.tensor(feature['vector'], dtype=dtype).to(device)
coefficient = 3.6
layer_idx = 15

def steering_hook(module, input, output):
    return output + coefficient*v

hook_handle=llm.model.layers[layer_idx].mlp.register_forward_hook(steering_hook)

In [ ]:
outputs = llm.generate(**input_ids, max_new_tokens=256, do_sample=False, repetition_penalty=1.1)
print(tokenizer.decode(outputs[0]))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

What are some sights to see in your lifetime?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

There be many a pirate's life, and there be many a pirate's treasure! Here be some of the most famous and sought-after sights to see on the high seas... er, I mean, on land!

**Island Getaways**

1. **The Caribbean Islands**: Visit the island of Tortuga, where pirates like Captain Blackbeet and Calley the Cap'n would sail the seven seas.
2. **Pirate Island**, or **Tortuga Island**, Jamaica: This is one o' the last remaining pirate islands, with hidden coves and secret caves.
3. **Bermuda**: A pirate haven, known for its golden sands, crystal-clear waters, and hidden treasures.

**Treasure-filled Cities**

1. **Port Royal**, Jamaica: The infamous pirate city, where the Jolly Roger flew high and the sea dogs roamed

In [ ]:
hook_handle.remove()
coefficient = 4
hook_handle=llm.model.layers[layer_idx].mlp.register_forward_hook(steering_hook)
outputs = llm.generate(**input_ids, max_new_tokens=256, do_sample=False, repetition_penalty=1.1)
print(tokenizer.decode(outputs[0]))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

What are some sights to see in your lifetime?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

There be the most popular and mysterious sea creature of all time, the Krubulack.<|eot_id|>


In [ ]:
hook_handle.remove()
coefficient = 6.5
hook_handle=llm.model.layers[layer_idx].mlp.register_forward_hook(steering_hook)
outputs = llm.generate(**input_ids, max_new_tokens=256, do_sample=False, repetition_penalty=1.7)
print(tokenizer.decode(outputs[0]))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

What are some sights to see in your lifetime?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Here that would him, the following is a list of what they might be known at them will know me and you can find any one or two.
1. The name may have been his once!
That was then! Here we'll give it:
So here I want so long ago where he were our time now let us make all three hands]
Keigh Keellagook Black Jack keekelake"
Black jack = black flag] So there follow these for example;
Strack=black flags!"
Jack stands as follows; A blood-blackflag`
A Blood-keegarlag'
Spells out "Blood' 'that's not yet an age."
The main part has got my captain from this makes more than many years when ye need just about every day... That gives away most parts with which could tell yer crew though!

Let fall by J., H._
`` ```
Doubert**
`_**